# Agent Demos — LaFrieda ERP Testbed

Run agent scenarios and observe tool-call traces.

**Requires**: `ANTHROPIC_API_KEY` set in `.env`

In [ ]:
import sys
sys.path.insert(0, '..')

from database.connection import get_session
from analysis.llm_client import LLMClient
from agents.tool_registry import ToolRegistry
from agents.agent_runner import AgentRunner
from agents.prompts.vendor_ops_agent import SYSTEM_PROMPT

session = get_session()
llm = LLMClient()

# Register tools
registry = ToolRegistry()
from agents.tools.query_database import TOOL_DEF as qd_tool
from agents.tools.reorder_suggestions import TOOL_DEF as rs_tool
from agents.tools.campaign_generator import TOOL_DEF as cg_tool
from agents.tools.alert_triggers import TOOL_DEF as at_tool
from agents.tools.dispute_handler import TOOL_DEF as dh_tool
from agents.tools.inventory_optimizer import TOOL_DEF as io_tool
from agents.tools.payment_optimizer import TOOL_DEF as po_tool

for tool in [qd_tool, rs_tool, cg_tool, at_tool, dh_tool, io_tool, po_tool]:
    registry.register(tool)

print(f'Registered {len(registry._tools)} tools')

In [ ]:
# Run the Vendor Ops agent with a spoilage question
runner = AgentRunner(llm, registry, SYSTEM_PROMPT)
result = runner.run("What inventory is at risk of expiring in the next 7 days?")

print(f"Tool calls: {result.tool_calls_count}")
print(f"\nFinal Answer:\n{result.final_answer}")

In [ ]:
# View the full trace
for step in result.trace:
    role = step.get('role', 'unknown')
    if role == 'tool_call':
        print(f"\n--- Tool Call: {step['name']} ---")
        print(f"Args: {step['arguments']}")
    elif role == 'tool_result':
        print(f"Result: {str(step['content'])[:500]}")
    elif role == 'assistant':
        print(f"\n--- Assistant ---")
        print(step.get('content', '')[:500])

In [ ]:
# Run the full scenario suite
from scenarios.vendor_scenarios import VENDOR_SCENARIOS
from scenarios.runner import run_all_scenarios

results = run_all_scenarios(VENDOR_SCENARIOS, SYSTEM_PROMPT, registry, llm)